In [1]:
cd ..

/home/credit-score/hainh/RecSys/hashtag-rec


/home/credit-score/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import json
import re
import pandas as pd
import textdistance as td
from keybert import KeyBERT
from collections import Counter
from keyphrase_vectorizers import KeyphraseCountVectorizer
from constant.twitter_constant import Dapps

ModuleNotFoundError: No module named 'textdistance'

## Pretrain models

In [3]:
# from flair.embeddings import TransformerDocumentEmbeddings

# roberta = TransformerDocumentEmbeddings('meta-llama/Meta-Llama-3-8B')
kw_model = KeyBERT(model='all-mpnet-base-v2')
#keywords = kw_model.extract_keywords(message, keyphrase_ngram_range=(0,1))
#kw_model1 = KeyBERT(model='multi-qa-mpnet-base-dot-v1')


In [4]:
# test
doc = "When Sebastian Thrun started working on self-driving cars at Google in 2007, few people outside of the company took him  seriously."
keywords = kw_model.extract_keywords(doc,vectorizer=KeyphraseCountVectorizer(), top_n=10, use_mmr=True)
print(keywords)


[('sebastian thrun', 0.3609), ('google', 0.2741), ('cars', 0.2353), ('self', 0.2316), ('few people', 0.2255), ('company', 0.17)]


## X data

In [19]:
def remove_URL(text):
    """Remove URLs from a text string."""
    return re.sub(r"http\S+", "http", text)


def get_hashtag(message):
    mess = re.split(r'\s+|\n|\[|\]', message)
    hashtags =[]
    for me in mess:
        if "#" in me:
            hashtags.append(me[1:])
    return hashtags


with open("data/protocols.json",'r') as rfile:
    protocols_data = json.load(rfile)

with open("data/influencers.json",'r') as rfile:
    influencers_data = json.load(rfile)

with open("data/pages.json",'r') as rfile:
    pages_data = json.load(rfile)


protocols_data.extend(pages_data)
protocols_data.extend(influencers_data)
hashtags = []
for da in protocols_data: 
    hashtag_info = dict()
    hashtag_info['_id'] = da['_id']
    hashtag_info['content'] = da['text']
    hashtag_info['author'] = da['author']
    hashtag_info['authorName'] = da['authorName']
    hashtag_info['timestamp'] =da['timestamp']
    hashtag_info['likes'] = da['likes']
    hashtag_info['retweetCounts'] = da['retweetCounts']
    hashtag_info['replyCounts'] = da['replyCounts']

    hashtags.append(hashtag_info)

df_hashtags = pd.DataFrame(hashtags).drop_duplicates().reset_index(drop=True)


In [22]:
df_hashtags

,_id,content,author,authorName,timestamp,likes,retweetCounts,replyCounts
0,1780597089329684920,🤖 #TRAVA GPT V2.0 IS OFFICIALLY LIVE FOR EVERY...,1387236181822164998,trava_finance,1713362402,76,25,8
1,1782771425805472049,🎉#ZEALY #SPRINTQUEST PHASE #05 HAS ENDED🔚\n\nH...,1387236181822164998,trava_finance,1713880804,35,8,11
2,1783038836932829291,📌 Don't forget to fill in your wallet address ...,1387236181822164998,trava_finance,1713944560,8,2,1
3,1783039340253442279,"Dear Trava Knight Fellows,\n\nJust a friendly ...",1387236181822164998,trava_finance,1713944680,11,3,0
4,1782756326273364171,📢 If you own an affected wallet after we stopp...,1387236181822164998,trava_finance,1713877204,36,10,2
...,...,...,...,...,...,...,...,...
18803,1775534443698253987,Holy smokes. It's happening. IBC for Ethereum.,334659208,TyreeRobinson,1712155373,30,4,1
18804,1775533954889850967,🎨 @Ethereum @IBCprotocol is live on https://t....,334659208,TyreeRobinson,1712155257,0,252,0
18805,1775388575497945309,".@orchai_protocol, a low-code DeFi platform pr...",334659208,TyreeRobinson,1712120596,0,18,0
18806,1774501443942117855,"Welcome, and keep in mind what you can stake $...",334659208,TyreeRobinson,1711909087,0,1,0


In [21]:
df_hashtags_trend = df_hashtags[(df_hashtags['timestamp']>= 1711929600)].reset_index(drop=True)
df_hashtags_trend


,_id,content,author,authorName,timestamp,likes,retweetCounts,replyCounts
0,1780597089329684920,🤖 #TRAVA GPT V2.0 IS OFFICIALLY LIVE FOR EVERY...,1387236181822164998,trava_finance,1713362402,76,25,8
1,1782771425805472049,🎉#ZEALY #SPRINTQUEST PHASE #05 HAS ENDED🔚\n\nH...,1387236181822164998,trava_finance,1713880804,35,8,11
2,1783038836932829291,📌 Don't forget to fill in your wallet address ...,1387236181822164998,trava_finance,1713944560,8,2,1
3,1783039340253442279,"Dear Trava Knight Fellows,\n\nJust a friendly ...",1387236181822164998,trava_finance,1713944680,11,3,0
4,1782756326273364171,📢 If you own an affected wallet after we stopp...,1387236181822164998,trava_finance,1713877204,36,10,2
...,...,...,...,...,...,...,...,...
11156,1776700333667848585,The alpha is $LAB @madscientists_x.\n\n@BackBo...,334659208,TyreeRobinson,1712433343,5,1,0
11157,1776652531331891417,$LAB @madscientists_x https://t.co/ksutXtl1uT,334659208,TyreeRobinson,1712421946,4,0,0
11158,1775534443698253987,Holy smokes. It's happening. IBC for Ethereum.,334659208,TyreeRobinson,1712155373,30,4,1
11159,1775533954889850967,🎨 @Ethereum @IBCprotocol is live on https://t....,334659208,TyreeRobinson,1712155257,0,252,0


In [23]:
def process_keywords(d, threshold):
    merged_dict = {}
    keys = list(d.keys())
    check_key = []
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            key1, key2 = keys[i], keys[j]
            if ([key1, key2] in check_key)| ([key2, key1] in check_key):
                continue
            else:
                distance = td.jaro_winkler(key1,key2)  
                if distance > threshold:
                    merged_key = key1
                    merged_value = d[key1] + d[key2]
                    merged_dict[merged_key] = merged_value
                    check_key.append([key1,key2])
                    del d[key1]
                    del d[key2]
    
    d.update(merged_dict)

    return d

In [8]:
def get_keywords(row):
    content = row['content']
    remove_url_content = remove_URL(content)
    keywords = kw_model.extract_keywords(remove_url_content, keyphrase_ngram_range=(1,1), top_n=10, use_mmr=True)
    keywords_lst = []            
    for key in keywords:
        if key[1] >=0.15:
            try:
                int(key[0])
            except Exception as e:
                keywords_lst.append(key[0])
    return keywords_lst

    
df_hashtags_trend['keyWords'] = df_hashtags_trend.apply(lambda row: get_keywords(row), axis=1)

def get_vector_keywords(row):
    content = row['content']
    try:
        remove_url_content = remove_URL(content)
        keywords = kw_model.extract_keywords(remove_url_content,vectorizer=KeyphraseCountVectorizer(), top_n=10, use_mmr=True)
        keywords_lst = []            
        for key in keywords:
            if key[1] >=0.1:
                try:
                    int(key[0])
                except Exception as e:
                    keywords_lst.append(key[0])
    except Exception as e:
        keywords_lst=[]
        return keywords_lst
    return keywords_lst

df_hashtags_trend['keyWordsVector'] = df_hashtags_trend.apply(lambda row: get_vector_keywords(row), axis=1)

In [ ]:
def get_vector_noun_keywords(row):
    content = row['content']
    try:
        remove_url_content = remove_URL(content)
        keywords = kw_model.extract_keywords(remove_url_content, vectorizer=KeyphraseCountVectorizer(pos_pattern='<N.*>'), top_n=10, use_mmr=True)
        keywords_lst = []            
        for key in keywords:
            if key[1] >=0.1:
                try:
                    int(key[0])
                except Exception as e:
                    keywords_lst.append(key[0])
    except Exception as e:
        keywords_lst=[]
        return keywords_lst
    return keywords_lst


df_hashtags_trend['keyWordsNounVector'] = df_hashtags_trend.apply(lambda row: get_vector_noun_keywords(row), axis=1)

In [11]:
df_hashtags_trend

,content,author,authorName,timestamp,likes,retweetCounts,replyCounts,keyWords
0,🔥NEW QUESTS IN #ZEALY PHASE 05🔥\n\n📣Our #Sprin...,1387236181822164998,trava_finance,1713448972,32,7,1,"[quests, xp, zealy, sprintquest, trava, gpt, l..."
1,🤖 #TRAVA GPT V2.0 IS OFFICIALLY LIVE FOR EVERY...,1387236181822164998,trava_finance,1713362402,67,19,7,"[travagpt, gpt, wallet, v2, limit]"
2,2 HOURS TO GO ‼️‼️,1387236181822164998,trava_finance,1713355202,49,15,5,[hours]
3,🫣[COMING #TOMORROW] #TRAVA #GPT V2.0🫣\n\nYou c...,1387236181822164998,trava_finance,1713276001,56,18,6,"[gpt, trava, v2, lending, wallet, data, tomorr..."
4,Join Nowww ‼️‼️‼️,1387236181822164998,trava_finance,1713268802,35,12,3,"[join, nowww]"
...,...,...,...,...,...,...,...,...
5207,The alpha is $LAB @madscientists_x.\n\n@BackBo...,334659208,TyreeRobinson,1712433343,5,1,0,"[backbone_labs, alpha, experiment, lab, madsci..."
5208,$LAB @madscientists_x https://t.co/ksutXtl1uT,334659208,TyreeRobinson,1712421946,4,0,0,"[http, madscientists_x, lab]"
5209,Holy smokes. It's happening. IBC for Ethereum.,334659208,TyreeRobinson,1712155373,30,4,1,"[ibc, ethereum, happening, holy]"
5210,🎨 @Ethereum @IBCprotocol is live on https://t....,334659208,TyreeRobinson,1712155257,0,252,0,"[interoperable, mainnet, http, ethereum, cosmo..."


In [12]:
df_hashtags_trend=df_hashtags_trend.reset_index(drop=True)
df_hashtags_trend = df_hashtags_trend.drop_duplicates(subset=['content', 'author'])
trava_following = Dapps(dapps_name='trava').following
trava_trend =  df_hashtags_trend[(df_hashtags_trend['authorName'].isin(trava_following))]
trava_keywords_lst = trava_trend.keyWords.tolist()
trava_keywords = [item for sublist in trava_keywords_lst for item in sublist]
trava_keywords_trending = Counter(trava_keywords)

In [13]:
trava_keywords_trending_process = dict(process_keywords(trava_keywords_trending,0.95))
##del trava_keywords_trending_process['http']

In [17]:
trava = {k: v for k, v in sorted(trava_keywords_trending_process.items(), key=lambda item: item[1], reverse=True)}
del trava['http']

In [18]:
def scale_dict_values(d):
    min_val = min(d.values())
    max_val = max(d.values())
    for key in d:
        d[key] = (d[key] - min_val) / (max_val - min_val)
    return d


In [19]:
import itertools
import numpy as np 
trava_scale = scale_dict_values(trava)
trava_30_items = dict(itertools.islice(trava_scale.items(), 30))

trava_30_items


{'usdt': 1.0,
 'token': 0.7741935483870968,
 'bybit': 0.7419354838709677,
 'bitcoin': 0.7096774193548387,
 'thecryptoark': 0.6451612903225806,
 'crypto': 0.6451612903225806,
 'funding': 0.6451612903225806,
 'join': 0.6129032258064516,
 'token2049': 0.5161290322580645,
 'share': 0.4838709677419355,
 'ai': 0.4838709677419355,
 'prize': 0.4838709677419355,
 'dubai': 0.45161290322580644,
 'event': 0.41935483870967744,
 'utc': 0.3548387096774194,
 'web3': 0.3548387096774194,
 'markets': 0.3548387096774194,
 'ready': 0.3548387096774194,
 'nft': 0.3225806451612903,
 'prices': 0.3225806451612903,
 'asset': 0.3225806451612903,
 'trading': 0.2903225806451613,
 'april': 0.25806451612903225,
 'decentralized': 0.25806451612903225,
 'launch': 0.25806451612903225,
 'raised': 0.25806451612903225,
 'zealy': 0.25806451612903225,
 'sprint': 0.25806451612903225,
 'platform': 0.25806451612903225,
 'pixelrealm': 0.25806451612903225}

In [42]:
update_keywords = dict()
update_keywords['_id'] = 'trava'
update_keywords['1713916800'] = trava_30_items

In [43]:
update_keywords

{'_id': 'trava',
 '1713916800': {'usdt': 1.0,
  'token': 0.7741935483870968,
  'bybit': 0.7419354838709677,
  'bitcoin': 0.7096774193548387,
  'thecryptoark': 0.6451612903225806,
  'crypto': 0.6451612903225806,
  'funding': 0.6451612903225806,
  'join': 0.6129032258064516,
  'token2049': 0.5161290322580645,
  'share': 0.4838709677419355,
  'ai': 0.4838709677419355,
  'prize': 0.4838709677419355,
  'dubai': 0.45161290322580644,
  'event': 0.41935483870967744,
  'utc': 0.3548387096774194,
  'web3': 0.3548387096774194,
  'markets': 0.3548387096774194,
  'ready': 0.3548387096774194,
  'nft': 0.3225806451612903,
  'prices': 0.3225806451612903,
  'asset': 0.3225806451612903,
  'trading': 0.2903225806451613,
  'april': 0.25806451612903225,
  'decentralized': 0.25806451612903225,
  'launch': 0.25806451612903225,
  'raised': 0.25806451612903225,
  'zealy': 0.25806451612903225,
  'sprint': 0.25806451612903225,
  'platform': 0.25806451612903225,
  'pixelrealm': 0.25806451612903225}}

In [44]:
cd /home/credit-score/hainh/RecSys/reputation-analysis/

/home/credit-score/hainh/RecSys/reputation-analysis


In [45]:
from databases.mongodb import MongoDB

In [46]:
MongoDB().update_data(name='trending_keywords',data=[update_keywords])